In [ ]:
# Staged by Agent Ops Board. PRIVATE notebook only.
AOB_RUN_ID = 'run-a57d336af9d3709f'
AOB_EVENTS_URL = 'https://evanescence-pgp-indicators-dietary.trycloudflare.com/api/events'
_AOB_KEY = '2f713cf087f35ef495d203e493d628e4c72ccbd676fa7bbdbf96edc34e6eb762'
import hashlib, hmac, json, time, urllib.request, uuid

def aob_emit(event_type, payload=None):
    """Send a signed event to the local Agent Ops Board through its public HTTPS URL."""
    body = json.dumps({"event_id": uuid.uuid4().hex, "run_id": AOB_RUN_ID,
                       "event_type": event_type, "ts": time.time(),
                       "payload": payload or {}}).encode("utf-8")
    timestamp = str(int(time.time()))
    signature = "sha256=" + hmac.new(_AOB_KEY.encode("utf-8"),
                                     timestamp.encode("ascii") + b"." + body,
                                     hashlib.sha256).hexdigest()
    for attempt in range(3):
        request = urllib.request.Request(
            AOB_EVENTS_URL, data=body, method="POST",
            headers={"Content-Type": "application/json",
                     "X-AOB-Timestamp": timestamp,
                     "X-AOB-Signature": signature})
        try:
            with urllib.request.urlopen(request, timeout=15) as response:
                response.read()
            return True
        except Exception as exc:
            # Do not print request headers / secrets; preserve validated GPU execution.
            print("[AOB EVENT] delivery failed for", event_type,
                  "attempt", attempt + 1, "error type", type(exc).__name__, flush=True)
            if attempt < 2:
                time.sleep(2 ** attempt)
    return False


In [ ]:
aob_emit("notebook_started", {"status": "running"})


# Kaggle GPU · Qwen3-4B Evaluation Auto-Restore
每次启动新 Session，设置 GPU + Internet 后，修改下方 `REPO_URL`，选择 **Run All**。环境文件是临时的；该 Notebook 会重新构建。

In [ ]:
from pathlib import Path
import os, subprocess, sys, time, requests

# === 修改为你的真实 GitHub 仓库地址（公开仓库）===
REPO_URL = "https://github.com/5tannerpl/agent-ops-board.git"
REPO_BRANCH =  "main"

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "agent-repo"
LLAMA_DIR = WORK / "llama.cpp"
MODEL_DIR = WORK / "models"
MODEL_FILE = MODEL_DIR / "qwen3-4b-instruct-2507-q8_0.gguf"
SERVER_BIN = LLAMA_DIR / "build/bin/llama-server"
LOG_FILE = WORK / "llama-server.log"
API = "http://127.0.0.1:8080"

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("请先把 REPO_URL 修改为真实的 GitHub Repo 地址。")
print("Repo:", REPO_URL)
subprocess.run(["nvidia-smi"], check=True)

## 1. 获取最新 GitHub 代码（存在则 pull，否则 clone）

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    args = ["git", "clone", "--depth", "1"]
    if REPO_BRANCH:
        args += ["--branch", REPO_BRANCH]
    args += [REPO_URL, str(REPO_DIR)]
    subprocess.run(args, check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)

## 2. 安装最小测试依赖
不安装全部生产系统依赖；避免不必要的 Kaggle 预装包冲突。

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "httpx", "requests", "huggingface_hub>=0.34,<1"], check=True)

## 3. 优先恢复编译 Cache Tesla T4 (sm_75) CUDA llama.cpp
优先恢复编译 Cache → 验证 llama-server → 不可用才重新编译

In [ ]:

# Step 3: Restore CUDA llama.cpp from Kaggle Dataset
# NO CLONE / NO CMAKE / NO REBUILD

from pathlib import Path
import os
import shutil
import subprocess

CACHE_ROOT = Path(
    "/kaggle/input/datasets/sillypanda5/"
    "agent-ops-llama-cuda-t4"
)

SOURCE_BIN = CACHE_ROOT / "bin"
LLAMA_DIR = Path("/kaggle/working/llama.cpp")
BIN_DIR = LLAMA_DIR / "build/bin"
SERVER_BIN = BIN_DIR / "llama-server"

assert (SOURCE_BIN / "llama-server").is_file(), (
    f"CUDA cache not found: {SOURCE_BIN}"
)

# Copy all binaries and shared libraries
BIN_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    SOURCE_BIN,
    BIN_DIR,
    dirs_exist_ok=True,
    symlinks=False
)

SERVER_BIN.chmod(0o755)

# Ensure versioned library aliases exist
for lib in BIN_DIR.glob("*.so.*.*.*"):
    prefix, version = lib.name.split(".so.", 1)
    major = version.split(".")[0]

    for alias in [
        f"{prefix}.so.{major}",
        f"{prefix}.so"
    ]:
        link = BIN_DIR / alias
        if not link.exists() and not link.is_symlink():
            link.symlink_to(lib.name)

# Runtime library path
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = (
    str(BIN_DIR) + ":" + env.get("LD_LIBRARY_PATH", "")
)

# Check missing dependencies
check = subprocess.run(
    ["ldd", str(SERVER_BIN)],
    capture_output=True,
    text=True,
    env=env
)

missing = [
    line for line in check.stdout.splitlines()
    if "not found" in line
]

if missing:
    raise RuntimeError(
        "Missing libraries:\n" + "\n".join(missing)
    )

# Validate executable
result = subprocess.run(
    [str(SERVER_BIN), "--version"],
    capture_output=True,
    text=True,
    env=env,
    timeout=30
)

print(result.stdout)
print(result.stderr)

assert result.returncode == 0, (
    "Cached llama-server failed to execute"
)

print("CUDA CACHE RESTORED — NO REBUILD")
print("SERVER_BIN:", SERVER_BIN)


## 4. 「优先加载 Kaggle Dataset 中的 Qwen3-4B 模型」


In [ ]:

# ============================================================
# Step 4: Resolve Qwen3-4B model from Kaggle Dataset
# NO inference test here
# ============================================================

from pathlib import Path

MODEL_NAME = "qwen3-4b-instruct-2507-q8_0.gguf"

# Priority 1: Kaggle Dataset
dataset_root = Path("/kaggle/input")

candidates = [
    p for p in dataset_root.rglob(MODEL_NAME)
    if p.is_file()
    and "agent-ops-qwen-model" in str(p)
]

# Priority 2: Local model
if not candidates:
    local_root = Path("/kaggle/working/models")

    if local_root.exists():
        candidates = list(local_root.rglob(MODEL_NAME))

if not candidates:
    raise FileNotFoundError(
        "Qwen GGUF not found in Kaggle Dataset or local cache"
    )

# Compatible with Step 5
MODEL_FILE = str(candidates[0])
MODEL_PATH = MODEL_FILE

print("Model source:", MODEL_FILE)
print(
    "Model size:",
    round(Path(MODEL_FILE).stat().st_size / 1024**3, 2),
    "GiB"
)

print("STEP 4 COMPLETE — MODEL LOCATED")


## 5. 启动或复用 llama-server，并检查健康状态

In [ ]:

# ============================================================
# Step 5: Start or reuse CUDA llama-server
# ============================================================

from pathlib import Path
import os
import subprocess
import time
import requests

API = "http://127.0.0.1:8080"
LOG_FILE = Path("/kaggle/working/llama-server.log")

# Required variables from previous cells
assert Path(SERVER_BIN).is_file(), "Run Step 3 first"
assert Path(MODEL_FILE).is_file(), "Run Step 4 first"

# Inherit environment from Step 3
server_env = env.copy()

def healthy():
    try:
        response = requests.get(
            API + "/health",
            timeout=3
        )
        return response.status_code == 200
    except requests.RequestException:
        return False

print("=== llama-server Startup ===")

if healthy():
    print("Existing llama-server is ready:", API)

else:
    print("Starting new CUDA llama-server...")
    print("Binary:", SERVER_BIN)
    print("Model:", MODEL_FILE)
    print("Log:", LOG_FILE)

    # Prevent repeated launches in same Notebook kernel
    old_server = globals().get("server")

    if (
        old_server is not None
        and old_server.poll() is None
    ):
        raise RuntimeError(
            "Previous llama-server process is still running "
            "but health check failed. Inspect logs first."
        )

    log_handle = open(LOG_FILE, "a", buffering=1)

    try:
        server = subprocess.Popen(
            [
                str(SERVER_BIN),
                "-m", str(MODEL_FILE),
                "--alias", "Qwen3-4B-Instruct-2507",
                "--host", "127.0.0.1",
                "--port", "8080",
                "-ngl", "99",
                "-c", "4096",
                "--device", "CUDA0"
            ],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=server_env
        )

        print("Started PID:", server.pid)

        # Wait for model initialization
        for attempt in range(90):
            if healthy():
                print("llama-server is ready")
                break

            if server.poll() is not None:
                raise RuntimeError(
                    f"llama-server exited with code "
                    f"{server.returncode}"
                )

            time.sleep(2)

        else:
            raise RuntimeError(
                "llama-server did not become ready "
                "within 180 seconds."
            )

    except Exception:
        print("\n=== Recent llama-server logs ===")

        if LOG_FILE.exists():
            lines = LOG_FILE.read_text(
                errors="replace"
            ).splitlines()

            print("\n".join(lines[-60:]))

        raise

    finally:
        log_handle.close()

# Final verification
response = requests.get(API + "/health", timeout=5)

print("\n=== Final Health Check ===")
print("HTTP:", response.status_code)
print("Response:", response.text[:500])

if response.status_code != 200:
    raise RuntimeError("llama-server is not ready")

print("\n=== GPU Status ===")
subprocess.run(["nvidia-smi"], check=True)

print("\nSERVER READY:", API)


## 6. 运行 GitHub 仓库里的测试
先运行原有 `workers.check_model`；如果存在 `modelcheck.py`，也会运行它。

In [ ]:

# ============================================================
# Step 6: Model Validation + HTTPS Event Reporting
#
# 1. Run workers.check_model (10 runs)
# 2. Capture real stdout/stderr
# 3. Send test results to external HTTPS webhook
# 4. Run modelcheck.py
# 5. Send final validation results
# ============================================================

import os
import sys
import time
import uuid
import subprocess
import requests

from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

REPO_DIR = Path("/kaggle/working/agent-repo")

API = "http://127.0.0.1:8080"


RUN_ID = str(uuid.uuid4())

MODEL_NAME = "Qwen3-4B-Instruct-2507"

# ------------------------------------------------------------
# Environment configuration
# ------------------------------------------------------------

env = os.environ.copy()

env.update({
    "AOB_MODEL_BACKEND": "llamacpp",
    "AOB_MODEL_URL": API,
    "AOB_MODEL_NAME": MODEL_NAME,
    "AOB_MODEL_TIMEOUT": "120"
})

# ------------------------------------------------------------
# HTTPS Event Reporter
# ------------------------------------------------------------

sent = 0
failed = 0

def send_event(event_type, data):
    global sent, failed
    # Keep the old Step 6 webhook success/failure accounting.
    ok = aob_emit(event_type, data)
    if ok:
        sent += 1
    else:
        failed += 1
    return ok


# ------------------------------------------------------------
# Validate environment
# ------------------------------------------------------------

assert REPO_DIR.is_dir(), (
    "Agent repository not found. Run Step 1 first."
)

health = requests.get(
    API + "/health",
    timeout=5
)

health.raise_for_status()

print("=== Step 6: Model Validation ===")
print("Run ID:", RUN_ID)
print("Model:", MODEL_NAME)
print("Backend:", "llamacpp")
print("Model server health:", health.status_code)

# ------------------------------------------------------------
# 1. Report validation started
# ------------------------------------------------------------

send_event(
    "model_validation_started",
    {
        "test_runs": 10,
        "status": "running"
    }
)

# ------------------------------------------------------------
# 2. Run workers.check_model
# ------------------------------------------------------------

check_file = REPO_DIR / "workers" / "check_model.py"

if check_file.is_file():

    print(
        "\n=== workers.check_model (10 runs) ===",
        flush=True
    )

    started = time.perf_counter()

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "workers.check_model",
            "--runs",
            "10"
        ],
        cwd=str(REPO_DIR),
        env=env,
        capture_output=True,
        text=True,
        timeout=300
    )

    duration = time.perf_counter() - started

    print(result.stdout, flush=True)

    if result.stderr:
        print(
            "=== workers.check_model stderr ==="
        )
        print(result.stderr, flush=True)

    validation_passed = result.returncode == 0

    # Report real execution results
    send_event(
        "worker_validation_completed",
        {
            "test_runs": 10,
            "duration_seconds": round(duration, 3),
            "exit_code": result.returncode,
            "status": (
                "passed"
                if validation_passed
                else "failed"
            ),
            "stdout": result.stdout[-6000:],
            "stderr": result.stderr[-2000:]
        }
    )

    if not validation_passed:
        raise RuntimeError(
            "workers.check_model validation failed"
        )

else:
    print(
        "Skip workers.check_model: file not found"
    )

# ------------------------------------------------------------
# 3. Run modelcheck.py
# ------------------------------------------------------------

modelcheck_file = None

for rel in [
    "modelcheck.py",
    "workers/modelcheck.py"
]:
    candidate = REPO_DIR / rel

    if candidate.is_file():
        modelcheck_file = rel
        break

if modelcheck_file:

    print(
        f"\n=== {modelcheck_file} ===",
        flush=True
    )

    started = time.perf_counter()

    result = subprocess.run(
        [
            sys.executable,
            modelcheck_file
        ],
        cwd=str(REPO_DIR),
        env=env,
        capture_output=True,
        text=True,
        timeout=180
    )

    duration = time.perf_counter() - started

    print(result.stdout, flush=True)

    if result.stderr:
        print("=== modelcheck stderr ===")
        print(result.stderr, flush=True)

    validation_passed = result.returncode == 0

    send_event(
        "modelcheck_completed",
        {
            "duration_seconds": round(duration, 3),
            "exit_code": result.returncode,
            "status": (
                "passed"
                if validation_passed
                else "failed"
            ),
            "stdout": result.stdout[-4000:],
            "stderr": result.stderr[-2000:]
        }
    )

    if not validation_passed:
        raise RuntimeError(
            "modelcheck.py failed"
        )

else:
    print(
        "Skip modelcheck.py: file not found"
    )

# ------------------------------------------------------------
# 4. Report all validations completed
# ------------------------------------------------------------

send_event(
    "model_validation_completed",
    {
        "status": "passed",
        "model": MODEL_NAME,
        "backend": "llamacpp",
        "api_health": 200
    }
)

print("\n======================================")
print("STEP 6 COMPLETE")
print("MODEL VALIDATION PASSED")
print("HTTPS EVENT TEST FINISHED")
print("RUN_ID:", RUN_ID)
print("======================================")


## 常见问题
- **新 Session**：Kaggle 临时磁盘会重置，重新 Run All 即可。
- **私有 GitHub Repo**：该 Notebook 的匿名 clone 不适用；需要自行配置 Kaggle Secrets + 只读 token，不要在 Cell 里明文写 token。
- **构建错误**：查看当前编译输出；`llama-server` 运行错误查看 `/kaggle/working/llama-server.log`。
- **测试程序位置**：如果你的文件不在上述默认路径，修改最后一个 Cell。
- **保存结果**：Session 结束前，将需要的测试报告导出到持久存储。